# 06 · Cabeças Wav-KAN: parâmetros, equivalência e tempo só da cabeça

**Sessão P3 · ambiente L4 · ~10 min (≈ 0,3 UC).** Não usa imagens nem gera cache: só tensores aleatórios.

| Parte | O que faz |
|---|---|
| 1 | contagem exata de parâmetros de (a)–(d) e W1–W5 (entrada/saída) e de W1/W2 (MLSD), com `assert` |
| 2 | equivalência numérica com o `WavKANLayer` do torch-conv-kan (clone no commit fixado) e com uma referência em NumPy, nas 3 wavelets, BN em treino e em avaliação, fp32, atol 1e-5 — na CPU e na GPU |
| 3 | tempo e memória **só da cabeça** na L4: fwd+bwd com lote 32, inferência com lotes 1 e 256 (eventos CUDA, 20 aquecimentos + 200 medidas); (a)–(d) entram como âncoras |
| 4 | horas e unidades previstas para cada W (entrada/saída e MLSD), a partir das rodadas reais de (d) no P1, do tempo da cabeça (parte 3) e da parte 7 do `01` (W1/W5 de ponta a ponta) |

O tempo de ponta a ponta de W1 e W5 já é medido na parte 7 do `01_calibracao_L4.ipynb` e não é repetido aqui.

Grava `resultados/benchmark_cabecas.json` (GPU, versões, hashes do protocolo e do plugin, tudo o que foi medido). O `07` lê a previsão daqui.

**Como rodar:** escolha a **L4** e clique em **Executar tudo**. Não precisa mudar nada. No fim o ambiente é desconectado sozinho (`DESLIGAR_AO_FIM`).

In [1]:
# ==== Configuração desta sessão ====
DIR_DRIVE = "/content/drive/MyDrive/Colab Notebooks/tcc-final"  # pasta tcc-final (conta padrão, authuser=0)
RAIZ_TRABALHO = "/content"   # onde o torch-conv-kan é clonado (disco local, temporário)
SHA_PROTOCOLO_ESPERADO = "b7cd9baca53edd96c855cca10914fc258efa8ddfd0937cc0d3e9572de5ef9e13"
SHA_PLUGIN_ESPERADO = "d02f6ca6f41f0530df0f0cc3886382a8c35311ac1ec812c1f65fa74e18d800ed"   # cabecas_wavkan.py
PERMITIR_RESERVA = False     # o tempo só vale na L4
USAR_COMPILE = False         # True mede também com torch.compile (opcional; +2–3 min)
N_AQUEC, N_MEDIDAS = 20, 200
DESLIGAR_AO_FIM = True       # desconecta o ambiente no fim (não gasta unidades parado)

In [2]:
# Dependências fixadas (as mesmas das rodadas; a KAN spline (d) entra como âncora)
!pip install -q "albumentations==2.0.8" "git+https://github.com/Blealtan/efficient-kan@7b6ce1c87f18c8bc90c208f6b494042344216b11"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
import sys, os, json, time
from pathlib import Path
import numpy as np, pandas as pd, torch
sys.path.insert(0, DIR_DRIVE)
import protocolo_comum as pc

pc.checar_sentinela_drive(DIR_DRIVE)   # PARA aqui se o Colab estiver logado em outra conta
assert pc.sha256_protocolo() == SHA_PROTOCOLO_ESPERADO, (
    "O protocolo_comum.py do Drive não é o publicado. Não rode nada; avise o P1.")
ARQ_PLUGIN = f"{DIR_DRIVE}/cabecas_wavkan.py"
assert pc.sha256_arquivo(ARQ_PLUGIN) == SHA_PLUGIN_ESPERADO, (
    "O cabecas_wavkan.py do Drive não é o publicado. Não rode nada; avise a sessão P3.")
pc.carregar_plugin(ARQ_PLUGIN)
cw = sys.modules["cabecas_wavkan"]
print("Dependências:", pc.checar_dependencias())
print("Protocolo", pc.PROTOCOLO_VERSAO, "| SHA-256", pc.sha256_protocolo()[:16], "… | plugin", SHA_PLUGIN_ESPERADO[:16], "…")
GPU = pc.exigir_gpu(permitir_reserva=PERMITIR_RESERVA)   # medições comparáveis só na L4
DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"
print("GPU:", GPU["nome"], "| comparável:", GPU["comparavel"])
R = Path(DIR_DRIVE) / "resultados"
ARQ = R / "benchmark_cabecas.json"
bench = {"quando": pc.agora_iso(), "gpu": GPU, "comparavel": GPU["comparavel"], "ambiente": pc.info_ambiente(),
         "protocolo_sha256": pc.sha256_protocolo(), "plugin_sha256": SHA_PLUGIN_ESPERADO, "plugin_versao": cw.VERSAO}
def salvar():
    pc.gravar_json_atomico(bench, ARQ)
salvar()

Dependências: {'albumentations': '2.0.8', 'efficient_kan_commit': '7b6ce1c87f18c8bc90c208f6b494042344216b11'}
Protocolo 1.0.0 | SHA-256 b7cd9baca53edd96 … | plugin d02f6ca6f41f0530 …
GPU: NVIDIA L4 | comparável: True


## 1. Parâmetros (contagem exata)

In [5]:
NATIVAS = {"a": "linear", "b": "mlp128", "c": "mlp_iso", "d": "kan"}
par = {"es": {}, "dist": {}}
for t, n in (("es", 2), ("dist", 3)):
    for k, cab in NATIVAS.items():
        p = sum(x.numel() for x in pc._CABECAS[cab](2048, n).parameters())
        par[t][k] = {"cabeca": p, "treinaveis_fase2": cw.BACKBONE_RESNET152 + p}
for t, v in cw.checar_parametros().items():   # assert: W1–W5 = 8202·h + 4 (E/S) e 8206·h + 6 (MLSD)
    par[t].update(v)
assert [par["es"][k]["cabeca"] for k in "abcd"] == [4098, 262530, 3935871, 3936000]
assert [par["dist"][k]["cabeca"] for k in "abcd"] == [6147, 262659, 3937791, 3937920]
bench["parametros"] = par
salvar()
tab_par = pd.DataFrame({t: {k: v["cabeca"] for k, v in d.items()} for t, d in par.items()}).rename(
    columns={"es": "cabeça (E/S)", "dist": "cabeça (MLSD)"})
tab_par["treináveis F2 (E/S)"] = [par["es"][k]["treinaveis_fase2"] for k in tab_par.index]
display(tab_par)
e = par["es"]
print(f"(d)/W1 = {e['d']['cabeca'] / e['W1']['cabeca']:.2f}× | (d)/W2 = {e['d']['cabeca'] / e['W2']['cabeca']:.1f}× | "
      f"W2 − (b) = {e['W2']['cabeca'] - e['b']['cabeca']:+d} | W5 − (d) = {e['W5']['cabeca'] - e['d']['cabeca']:+d}")

,cabeça (E/S),cabeça (MLSD),treináveis F2 (E/S)
a,4098,6147.0,58147906
b,262530,262659.0,58406338
c,3935871,3937791.0,62079679
d,3936000,3937920.0,62079808
W1,1049860,1050374.0,59193668
W2,262468,262598.0,58406276
W3,1049860,NaN,59193668
W4,1049860,NaN,59193668
W5,3936964,NaN,62080772


(d)/W1 = 3.75× | (d)/W2 = 15.0× | W2 − (b) = -62 | W5 − (d) = +964


## 2. Equivalência com o torch-conv-kan e com a referência NumPy

In [6]:
DIR_TCK = Path(RAIZ_TRABALHO) / "torch-conv-kan"
print("torch-conv-kan @", pc.clonar_fixado(cw.TORCH_CONV_KAN_URL, cw.TORCH_CONV_KAN_COMMIT, DIR_TCK))
eq = {}
for disp in dict.fromkeys(["cpu", DISPOSITIVO]):
    for lote, i, o in ((16, 64, 24), (32, 2048, 128)):
        r = cw.checar_equivalencia(DIR_TCK, dispositivo=disp, lote=lote, in_f=i, out_f=o)
        eq[f"{disp}_{i}x{o}"] = r
        print(f"{disp:4s} [{i}->{o}] máx. diferença {r['max_dif']:.2e} -> {'OK' if r['ok'] else 'FALHOU'}")
bench["equivalencia"] = eq
salvar()
display(pd.DataFrame(eq[f"{DISPOSITIVO}_2048x128"]["camadas"]))
assert all(r["ok"] for r in eq.values()), "A Wav-KAN diverge da referência: NÃO rode o 07; avise a sessão P3."

torch-conv-kan @ b412a3b839e35d0ddce378f833c2e6aa258cc251
cpu  [64->24] máx. diferença 6.60e-07 -> OK
cpu  [2048->128] máx. diferença 1.45e-06 -> OK
cuda [64->24] máx. diferença 6.91e-07 -> OK
cuda [2048->128] máx. diferença 1.23e-06 -> OK


,wavelet,modo_bn,max_dif_numpy,max_dif_torch_conv_kan,max_dif_grad_torch_conv_kan
0,mexican_hat,treino,7.843184e-07,0.0,0.0
1,mexican_hat,aval,6.563027e-07,0.0,NaN
2,dog,treino,1.233048e-06,0.0,0.0
3,dog,aval,8.366463e-07,0.0,NaN
4,morlet,treino,8.628123e-07,0.0,0.0
5,morlet,aval,6.420396e-07,0.0,NaN


## 3. Tempo e memória só da cabeça (L4, tensores aleatórios)

In [7]:
torch.backends.cudnn.benchmark = True
CABS = {"a": "linear", "b": "mlp128", "c": "mlp_iso", "d": "kan", **{w: w for w in cw.W}}
NOMES = {"a": "(a) Linear", "b": "(b) MLP-128", "c": "(c) MLP-iso", "d": "(d) KAN spline",
         **{w: f"{w} Wav-KAN {cw.ROTULO_WAVELET[v[0]]} h={v[1]}" for w, v in cw.W.items()}}
linhas = []
for k, nome in CABS.items():
    for modo, lote in (("fwd_bwd", 32), ("inferencia", 1), ("inferencia", 256)):
        for comp in ([False, True] if USAR_COMPILE else [False]):
            torch.manual_seed(0)
            cab = pc._CABECAS[nome](2048, 2)
            alvo = cab.como_sequencial() if hasattr(cab, "como_sequencial") else cab
            if comp:
                alvo = torch.compile(alvo)
            m = cw.medir_cabeca(alvo, 2048, lote, modo, N_AQUEC, N_MEDIDAS, DISPOSITIVO)
            linhas.append({"cabeca": k, "nome": NOMES[k], "compile": comp, **m})
            del cab, alvo
            if DISPOSITIVO == "cuda":
                torch.cuda.empty_cache()
    print("medido:", NOMES[k])
tab_cab = pd.DataFrame(linhas)
bench["so_cabeca"] = linhas
salvar()
vis = tab_cab[~tab_cab["compile"]].assign(conf=lambda d: d["modo"] + " lote " + d["lote"].astype(str))
display(vis.pivot_table(index="nome", columns="conf", values="ms_media", sort=False).round(3))
display(vis.pivot_table(index="nome", columns="conf", values="ms_dp", sort=False).round(3).add_suffix(" (dp)"))
display(vis[vis.modo == "fwd_bwd"].set_index("nome")[["mem_pico_mb"]].round(1))

medido: (a) Linear
medido: (b) MLP-128
medido: (c) MLP-iso
medido: (d) KAN spline
medido: W1 Wav-KAN mexhat h=128
medido: W2 Wav-KAN mexhat h=32
medido: W3 Wav-KAN dog h=128
medido: W4 Wav-KAN morlet h=128
medido: W5 Wav-KAN mexhat h=480


conf,fwd_bwd lote 32,inferencia lote 1,inferencia lote 256
nome,,,
(a) Linear,0.485,0.059,0.067
(b) MLP-128,0.729,0.120,0.136
(c) MLP-iso,0.734,0.122,0.237
(d) KAN spline,3.297,1.535,5.483
W1 Wav-KAN mexhat h=128,8.838,0.857,24.421
W2 Wav-KAN mexhat h=32,2.952,0.853,6.192
W3 Wav-KAN dog h=128,7.596,0.809,19.762
W4 Wav-KAN morlet h=128,8.548,0.814,22.039
W5 Wav-KAN mexhat h=480,38.357,0.848,90.788


conf,fwd_bwd lote 32 (dp),inferencia lote 1 (dp),inferencia lote 256 (dp)
nome,,,
(a) Linear,0.026,0.007,0.014
(b) MLP-128,0.023,0.010,0.012
(c) MLP-iso,0.031,0.013,0.010
(d) KAN spline,0.179,0.092,0.018
W1 Wav-KAN mexhat h=128,0.053,0.015,0.032
W2 Wav-KAN mexhat h=32,0.135,0.016,0.012
W3 Wav-KAN dog h=128,0.030,0.020,0.049
W4 Wav-KAN morlet h=128,0.037,0.018,0.024
W5 Wav-KAN mexhat h=480,0.037,0.025,0.062


,mem_pico_mb
nome,
(a) Linear,0.0
(b) MLP-128,0.0
(c) MLP-iso,0.2
(d) KAN spline,27.3
W1 Wav-KAN mexhat h=128,223.3
W2 Wav-KAN mexhat h=32,55.8
W3 Wav-KAN dog h=128,223.3
W4 Wav-KAN morlet h=128,255.3
W5 Wav-KAN mexhat h=480,837.2


## 4. Horas e unidades previstas para cada W

minutos por dobra(W) = minutos de treino por dobra de (d) **nas rodadas reais do P1** + 60 épocas × lotes por época × (tempo fwd+bwd da cabeça W − o de (d)) + 0,3 min (teste e checkpoints). Para W1 e W5 há também a estimativa de ponta a ponta da parte 7 do `01`; vale a maior das duas. Unidades = horas × taxa da L4 registrada no `01`.

In [8]:
cal = json.loads((R / "calibracao_l4.json").read_text())
TAXA = float((cal.get("unidades_por_hora") or {}).get("valor") or 1.54)
dob = pc.ler_dobras(Path(DIR_DRIVE) / "dados" / pc.ARQUIVO_DOBRAS)
LOTES = {"es": float(np.mean([int((dob["fold"] != k).sum()) // 32 for k in range(pc.N_DOBRAS)])),
         "dist": float(np.mean([int(((dob["fold"] != k) & (dob["y_es"] == 0)).sum()) // 32 for k in range(pc.N_DOBRAS)]))}
EXP_P1 = {"es": "ablacao_cabeca", "dist": "mlsd_cabeca"}
CFG_D = {"es": pc.CONFIGS_EXP1["d"], "dist": "mlsd__resnet152__kan"}
CFGS_W = {"es": cw.CONFIGS_ES, "dist": cw.CONFIGS_MLSD}

def runs_p1(exp):
    arq = R / exp / "runs.csv"
    r = pd.read_csv(arq) if arq.exists() else pc.ler_resultados(R, exp)
    return r[~r["teste_rapido"].astype(bool)] if len(r) else r

fb = tab_cab[(tab_cab.modo == "fwd_bwd") & ~tab_cab["compile"]].set_index("cabeca")["ms_media"]
e2e = cal.get("wavkan", {})
est_d = cal["vazao"][pc.CONFIGS_EXP1["d"]]["estimativa"]["min_por_dobra"]
prev, linhas = {"taxa_uc_h": TAXA, "lotes_por_epoca": LOTES}, []
for t in ("es", "dist"):
    r = runs_p1(EXP_P1[t])
    rd = r[(r["config"] == CFG_D[t]) & r["gpu"].fillna("").str.contains("L4")]
    assert len(rd), f"Não achei as rodadas de (d) em resultados/{EXP_P1[t]}"
    min_d = float(rd["treino_min"].mean())
    prev[t] = {"_ref_d_treino_min_por_dobra": min_d}
    for w, cfg in CFGS_W[t].items():
        m_cab = min_d + 60 * LOTES[t] * (fb[w] - fb["d"]) / 60000
        m_e2e = min_d * e2e[w]["estimativa"]["min_por_dobra"] / est_d if w in e2e else None
        m = max(m_cab, m_e2e or 0) + 0.3
        prev[t][cfg] = {"id": w, "min_por_dobra": m, "uc_por_dobra": m / 60 * TAXA, "horas_por_semente": 5 * m / 60,
                        "uc_por_semente": 5 * m / 60 * TAXA, "min_pela_cabeca": m_cab, "min_pelo_01": m_e2e}
        linhas.append({"tarefa": t, "id": w, "config": cfg, "min/dobra": m, "pelo tempo da cabeça": m_cab,
                       "pelo 01 (ponta a ponta)": m_e2e, "h por semente": 5 * m / 60, "UC por semente": 5 * m / 60 * TAXA})
ETAPAS = [("07 principal, entrada/saída (W1, W2 × 42/43/44)", "es", ["W1", "W2"], 3),
          ("07 MLSD (W1, W2 × 42/43/44)", "dist", ["W1", "W2"], 3),
          ("07 sensibilidade (W3, W4, W5 × 42)", "es", ["W3", "W4", "W5"], 1)]
tot = []
for nome, t, ids, n_sem in ETAPAS:
    h = sum(prev[t][CFGS_W[t][w]]["horas_por_semente"] for w in ids) * n_sem + 5 / 60
    tot.append({"etapa": nome, "horas_L4": round(h, 2), "UC": round(h * TAXA, 1)})
prev["etapas"] = tot
bench["previsao"] = prev
salvar()
print(f"Taxa da L4: {TAXA} UC/h | lotes por época: {LOTES} | (d) no P1: "
      f"{prev['es']['_ref_d_treino_min_por_dobra']:.1f} min/dobra (E/S), {prev['dist']['_ref_d_treino_min_por_dobra']:.1f} (MLSD)")
display(pd.DataFrame(linhas).round(2))
display(pd.DataFrame(tot))
print("Gravado em", ARQ)

Taxa da L4: 1.54 UC/h | lotes por época: {'es': 63.6, 'dist': 46.8} | (d) no P1: 8.5 min/dobra (E/S), 6.5 (MLSD)


,tarefa,id,config,min/dobra,pelo tempo da cabeça,pelo 01 (ponta a ponta),h por semente,UC por semente
0,es,W1,resnet152__wavkan_mexhat_h128,9.14,8.84,8.66,0.76,1.17
1,es,W2,resnet152__wavkan_mexhat_h32,8.77,8.47,NaN,0.73,1.12
2,es,W3,resnet152__wavkan_dog_h128,9.06,8.76,NaN,0.76,1.16
3,es,W4,resnet152__wavkan_morlet_h128,9.12,8.82,NaN,0.76,1.17
4,es,W5,resnet152__wavkan_mexhat_h480,11.17,10.72,10.87,0.93,1.43
5,dist,W1,mlsd__resnet152__wavkan_mexhat_h128,7.02,6.72,6.59,0.58,0.90
6,dist,W2,mlsd__resnet152__wavkan_mexhat_h32,6.74,6.44,NaN,0.56,0.87


,etapa,horas_L4,UC
0,"07 principal, entrada/saída (W1, W2 × 42/43/44)",4.56,7.0
1,"07 MLSD (W1, W2 × 42/43/44)",3.52,5.4
2,"07 sensibilidade (W3, W4, W5 × 42)",2.53,3.9


Gravado em /content/drive/MyDrive/Colab Notebooks/tcc-final/resultados/benchmark_cabecas.json


⚠️ Se `DESLIGAR_AO_FIM = True`, a célula abaixo desconecta o ambiente (não gasta unidades parado).

In [9]:
if DESLIGAR_AO_FIM:
    from google.colab import runtime
    runtime.unassign()